## 1. Load parametric description of the line

In [45]:
import numpy
import sys
import meshio
import vtk
from scipy.interpolate import BSpline
from scipy.spatial import cKDTree
from scipy.optimize import minimize
from collections import defaultdict
from scipy.interpolate import UnivariateSpline

sys.path.insert(0, "./src")  # add Folder_2 path to search list


h = 1.0
extension_factor = 0.08

# [ 4   5     6     7]
# [LAT, MED, INF, SUP]


id = 4
filename = "lat_h"+str(h).replace(".","_")+"_extended"

base_mesh = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/re-meshed_alt-1_h_1-0/"
base_centerline = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/centerlines/"

mesh = meshio.read(base_mesh + filename +".inp")
points = mesh.points

cells = mesh.cells_dict["tetra"]
centers = points[cells].mean(axis=1)  # shape: (n_cells, 3)


In [46]:
# -----------------------------------------------------------------------------
# 2. Load Spline Parametric Data & Mesh
# -----------------------------------------------------------------------------
# Load the parametric .npz array created earlier
data = numpy.load(base_centerline + "centerline_id_" + str(id) + ".npz")

# Reconstruct UnivariateSplines directly from tck tuples
sx = UnivariateSpline._from_tck((data["tx"], data["cx"], int(data["kx"])))
sy = UnivariateSpline._from_tck((data["ty"], data["cy"], int(data["ky"])))
sz = UnivariateSpline._from_tck((data["tz"], data["cz"], int(data["kz"])))


In [47]:
# -----------------------------------------------------------------------------
# 3. Fast Spline Evaluation Functions
# -----------------------------------------------------------------------------
def centerline_point(u):
    return numpy.array([sx(u), sy(u), sz(u)])

def centerline_tangent(u):
    # Evaluates exact analytical derivatives dx/du, dy/du, dz/du
    return numpy.array([sx.derivative()(u), sy.derivative()(u), sz.derivative()(u)])

# Extract knot boundaries correctly using .get_knots()
knots_x = sx.get_knots()
knots_y = sy.get_knots()
knots_z = sz.get_knots()

u_domain_min = min(knots_x[0], knots_y[0], knots_z[0])
u_domain_max = max(knots_x[-1], knots_y[-1], knots_z[-1])

u_samples = numpy.linspace(u_domain_min, u_domain_max, 500)
sample_points = numpy.column_stack([sx(u_samples), sy(u_samples), sz(u_samples)])
spline_tree = cKDTree(sample_points)

def closest_u(center):
    """Finds parameter u on the spline closest to a 3D centroid point."""
    # 1. Fast initial guess via KDTree
    _, idx = spline_tree.query(center)
    u0 = u_samples[idx]
    
    # 2. Local search bound around guess
    step = (u_samples[1] - u_samples[0]) * 2.0
    u_min = max(u_domain_min, u0 - step)
    u_max = min(u_domain_max, u0 + step)
    
    def objective(u):
        pt = centerline_point(u[0])
        return numpy.sum((pt - center)**2)
    
    res = minimize(objective, [u0], bounds=[(u_min, u_max)], method='L-BFGS-B')
    return res.x[0]

In [48]:
# -----------------------------------------------------------------------------
# 4. Extract Surface Triangles and Calculate Surface Normals
# -----------------------------------------------------------------------------
def extract_surface_triangles(cells):
    face_dict = defaultdict(int)
    for tet in cells:
        faces = [
            tuple(sorted([tet[0], tet[1], tet[2]])),
            tuple(sorted([tet[0], tet[1], tet[3]])),
            tuple(sorted([tet[0], tet[2], tet[3]])),
            tuple(sorted([tet[1], tet[2], tet[3]])),
        ]
        for f in faces:
            face_dict[f] += 1
    # Boundary faces belong to only one tetrahedron
    surface_faces = [list(f) for f, count in face_dict.items() if count == 1]
    return numpy.array(surface_faces)

surface_faces = extract_surface_triangles(cells)
surface_points = points[surface_faces]  # shape: (n_faces, 3, 3)

def triangle_normals(tri_pts):
    v0 = tri_pts[:, 0, :]
    v1 = tri_pts[:, 1, :]
    v2 = tri_pts[:, 2, :]
    n = numpy.cross(v1 - v0, v2 - v0)
    norms = numpy.linalg.norm(n, axis=1, keepdims=True)
    return n / numpy.maximum(norms, 1e-12)

surface_normals = triangle_normals(surface_points)
surface_centers = surface_points.mean(axis=1)

# KD-Tree to project closest boundary normal onto internal volume elements
surface_tree = cKDTree(surface_centers)

# -----------------------------------------------------------------------------
# 5. Compute Tangent (T) and Normal (N) Vectors for Every Tetrahedron Cell
# -----------------------------------------------------------------------------
T_list = []
N_list = []

for c in centers:
    # --- Tangent Vector ---
    u = closest_u(c)
    T = centerline_tangent(u)
    T_norm = numpy.linalg.norm(T)
    if T_norm > 1e-12:
        T = T / T_norm
    T_list.append(T)
    
    # --- Surface Normal Vector ---
    _, idx = surface_tree.query(c)
    N = surface_normals[idx]
    N_list.append(N)

T_array = numpy.array(T_list)
N_array = numpy.array(N_list)

print(f"Calculated T vectors: {T_array.shape}")
print(f"Calculated N vectors: {N_array.shape}")

# -----------------------------------------------------------------------------
# 6. Export VTU Mesh and CSV Format
# -----------------------------------------------------------------------------
# 1. Export .vtu for visualization
cell_data = {
    "Tangent": [T_array],
    "Normal": [N_array]
}

vtk_mesh = meshio.Mesh(
    points=mesh.points,  # Original spatial mesh points
    cells=[("tetra", cells)],
    cell_data=cell_data
)

vtu_filename = base_mesh + filename + "__volume-with_direction.vtu"
meshio.write(vtu_filename, vtk_mesh)
print(f"--> Saved VTU Mesh: {vtu_filename}")

# 2. Export CSV for numerical analysis / FEA setup
n_cells = len(cells)
element_ids = numpy.arange(1, n_cells + 1)

csv_data = numpy.column_stack([
    element_ids,
    T_array,
    N_array
])

header = "element_id,Tx,Ty,Tz,Nx,Ny,Nz"
csv_filename = base_mesh + filename + "_vectors_tangent_normal.csv"

numpy.savetxt(
    csv_filename, 
    csv_data, 
    fmt=["%d", "%.6f", "%.6f", "%.6f", "%.6f", "%.6f", "%.6f"], 
    delimiter=",", 
    header=header, 
    comments=''
)
print(f"--> Saved CSV Vectors: {csv_filename}")

Calculated T vectors: (9178, 3)
Calculated N vectors: (9178, 3)
--> Saved VTU Mesh: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/re-meshed_alt-1_h_1-0/lat_h1_0_extended__volume-with_direction.vtu
--> Saved CSV Vectors: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/re-meshed_alt-1_h_1-0/lat_h1_0_extended_vectors_tangent_normal.csv
